# 04 — TVG Construction (multi-city)

**Step A (per city, one-time):** fetch buildings + street network across
EACH CITY'S OWN study-area bounding box, precompute everything that
doesn't depend on any individual incident (building shape metrics,
building-type vocab, highway-type vocab, betweenness centrality,
orientation entropy), build that city's shared STRtree, cache to disk
under a per-city subdirectory of the combined interim tree
(`interim/osm_cache/{city}/`) -- buildings/streets are entirely
different per city, so this step cannot be shared across cities the way
SVG's per-point processing can be.

**Step B (per point, checkpointed):** road-projection, isovist
ray-casting, node/edge construction, save graph + QC map -- looped per
city (each city's points need that city's own cache/graph/buildings),
writing into the SAME combined `processed/tvg_graphs/` directory as
every other city (globally-unique, city-prefixed `point_id` means no
per-city subfolder is needed for the graphs themselves).

**Revised graph-construction rules** (see `src/tvg_builder.py` /
`configs/tvg_schema.yaml`):
- `intersection` nodes are now STRICTLY isovist-covered -- the nearest
  road edge's endpoints (u/v) are no longer force-included when they
  fall outside the isovist. `on_segment` connects to whichever of u/v
  actually qualify, so it can now carry 0, 1, or 2 edges instead of
  always exactly 2.
- `adjacent` (building-building) is now a k-nearest-neighbor graph
  (`adjacent_k_nearest`, default 5) with real centroid distance (meters)
  as the edge feature, replacing the old full clique with a constant
  flag.
- `crash_history` peer matching now also requires matching `city`, not
  just matching `fold_rep{r}` -- fold indices are per-city cluster ids
  (0..k-1), NOT globally unique across cities, so without this a Bogor
  point and a Somerville point sharing the same fold number would have
  been treated as spatial peers.

Uses `src/geo_utils.py`, `src/osm_fetch.py`, `src/isovist.py`,
`src/tvg_builder.py`, `src/tvg_visualize.py`, `src/manifest.py`.
CPU is sufficient.

In [ ]:
# ── Clone/update repo, mount Drive ──────────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q osmnx geopandas shapely pyproj networkx torch_geometric tqdm pyyaml pandas numpy matplotlib

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/tvg_schema.yaml") as f:
    tvg_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]

INTERIM_DIR = Path(paths_cfg["interim_dir"])
PROCESSED_DIR = Path(paths_cfg["processed_dir"])

# TVG graphs + QC maps are COMBINED across cities (globally-unique,
# city-prefixed point_id means no per-city subfolder is needed for
# these) -- only the OSM cache below is per-city, since buildings/streets
# genuinely differ per city.
TVG_OUT_DIR = PROCESSED_DIR / "tvg_graphs"
VIZ_OUT_DIR = INTERIM_DIR / "tvg_visualizations"
LOG_PATH = INTERIM_DIR / "tvg_construction_log.csv"

TVG_OUT_DIR.mkdir(parents=True, exist_ok=True)
VIZ_OUT_DIR.mkdir(parents=True, exist_ok=True)

ADJACENT_K_NEAREST = tvg_cfg.get("adjacent_k_nearest", 5)
print(f"Cities: {CITIES}")
print(f"Isovist radius: {tvg_cfg['isovist_radius_m']}m | crash_history: {tvg_cfg['crash_history_threshold_m']}m | "
      f"N_RAYS: {tvg_cfg['n_rays']} | adjacent_k_nearest: {ADJACENT_K_NEAREST}")

In [ ]:
import manifest
import geo_utils
import osm_fetch
import isovist as iso
import tvg_builder
import tvg_visualize

In [ ]:
# ── STEP A: one-time fetch + precompute, PER CITY (skipped entirely
#    for any city whose cache already exists) ──────────────────────
from tqdm.auto import tqdm

city_layers = {}  # city -> dict of everything Step B needs for that city

for city in CITIES:
    city_cache_dir = INTERIM_DIR / "osm_cache" / city

    if osm_fetch.cache_exists(city_cache_dir):
        print(f"[{city}] ✅ OSM cache already exists — loading, skipping re-fetch/re-compute.")
        (buildings_gdf, building_type_vocab, highway_vocab, G,
         betweenness, orientation_entropy, utm_crs) = osm_fetch.load_cache(city_cache_dir)
    else:
        print(f"[{city}] No cache found — fetching and precomputing (runs once)...")
        boundary_geojson = paths_cfg["per_city"][city]["boundary_geojson"]

        with tqdm(total=6, desc=f"[{city}] Step A") as pbar:
            buildings_gdf, G, utm_crs = osm_fetch.fetch_study_area_layers(
                boundary_geojson, tvg_cfg["bbox_padding_m"], tvg_cfg["network_type"]
            )
            pbar.set_description("Fetched buildings + streets"); pbar.update(1)

            buildings_gdf = osm_fetch.compute_building_shape_metrics(buildings_gdf)
            pbar.set_description("Computed building shape metrics"); pbar.update(1)

            buildings_gdf, building_type_vocab = osm_fetch.build_building_type_vocab(buildings_gdf)
            pbar.set_description("Built building-type vocab"); pbar.update(1)

            highway_vocab = osm_fetch.build_highway_vocab(G)
            pbar.set_description("Built highway-type vocab"); pbar.update(1)

            betweenness, orientation_entropy = osm_fetch.compute_intersection_metrics(G)
            pbar.set_description("Computed betweenness + orientation entropy"); pbar.update(1)

            osm_fetch.save_cache(city_cache_dir, buildings_gdf, building_type_vocab, highway_vocab,
                                  G, betweenness, orientation_entropy, utm_crs)
            pbar.set_description("Cached to disk"); pbar.update(1)

    # STRtree isn't picklable -- cheap to rebuild once per session from
    # the cached buildings layer, same rationale as the old single-city version.
    tree, boundaries = osm_fetch.build_building_strtree(buildings_gdf)
    buildings_sindex = buildings_gdf.sindex  # for tvg_visualize.py's fast window queries

    city_layers[city] = dict(
        buildings_gdf=buildings_gdf, building_type_vocab=building_type_vocab,
        highway_vocab=highway_vocab, G=G, betweenness=betweenness,
        orientation_entropy=orientation_entropy, utm_crs=utm_crs,
        tree=tree, boundaries=boundaries, buildings_sindex=buildings_sindex,
    )
    print(f"[{city}] Buildings: {len(buildings_gdf)} | Street nodes: {len(G.nodes)} | "
          f"Building types: {len(building_type_vocab)} | Highway types: {len(highway_vocab)}\n")

In [ ]:
# ── Load reconciled points from 01, project EACH CITY'S rows to ITS OWN
#    UTM CRS (needed for fast peer-incident distance lookups without
#    re-projecting per call -- one city's projection is meaningless for
#    another city's points). ──────────────────────────────────────
import pandas as pd
import geopandas as gpd
import numpy as np

reconciled = pd.read_parquet(INTERIM_DIR / "reconciled_points.parquet")
reconciled["_utm_x"] = np.nan
reconciled["_utm_y"] = np.nan

for city in CITIES:
    utm_crs = city_layers[city]["utm_crs"]
    city_mask = (reconciled["city"] == city).values
    city_pts = gpd.GeoDataFrame(
        reconciled[city_mask],
        geometry=gpd.points_from_xy(reconciled.loc[city_mask, "input_lon"], reconciled.loc[city_mask, "input_lat"]),
        crs="EPSG:4326",
    ).to_crs(utm_crs)
    reconciled.loc[city_mask, "_utm_x"] = city_pts.geometry.x.values
    reconciled.loc[city_mask, "_utm_y"] = city_pts.geometry.y.values

all_point_ids = reconciled["point_id"].tolist()
pending = manifest.pending_items(all_point_ids, TVG_OUT_DIR, ext=".pt")
print(f"Total points (all cities): {len(all_point_ids)}  |  Already done: {len(all_point_ids) - len(pending)}  |  Pending: {len(pending)}")

In [ ]:
# ── STEP B: main per-point loop, PER CITY (each city needs its own
#    cache/graph/buildings), checkpointed and resumable ───────────
import torch

row_lookup = reconciled.set_index("point_id", drop=False)

for city in CITIES:
    layers = city_layers[city]
    city_pending = [pid for pid in pending if row_lookup.loc[pid, "city"] == city]
    if not city_pending:
        print(f"[{city}] nothing pending, skipping.")
        continue

    for point_id in tqdm(city_pending, desc=f"[{city}] Building TVG"):
        try:
            incident_row = row_lookup.loc[point_id]

            data, meta = tvg_builder.process_incident(
                incident_row, reconciled, layers["buildings_gdf"], layers["building_type_vocab"],
                layers["highway_vocab"], layers["G"], layers["betweenness"], layers["orientation_entropy"],
                layers["tree"], layers["boundaries"], layers["utm_crs"],
                isovist_radius_m=tvg_cfg["isovist_radius_m"],
                n_rays=tvg_cfg["n_rays"],
                crash_history_threshold_m=tvg_cfg["crash_history_threshold_m"],
                adjacent_k_nearest=ADJACENT_K_NEAREST,
            )

            torch.save(data, TVG_OUT_DIR / f"{point_id}.pt")

            peer_xy = [
                (row_lookup.loc[pid, "_utm_x"], row_lookup.loc[pid, "_utm_y"])
                for pid in meta["peer_point_ids"]
            ]
            fig = tvg_visualize.render_tvg_overlay(
                meta["origin_xy"], int(incident_row["label"]), meta["polygon"],
                meta["included_building_ids"], layers["buildings_gdf"], layers["buildings_sindex"], peer_xy,
                G=layers["G"], data=data, included_intersections=meta["included_intersections"],
                u=meta["u"], v=meta["v"],
            )
            tvg_visualize.save_overlay(fig, VIZ_OUT_DIR, point_id)

            manifest.append_log(LOG_PATH, point_id, "tvg_construction", "ok")

        except Exception as e:
            manifest.append_log(LOG_PATH, point_id, "tvg_construction", "error", str(e))
            tqdm.write(f"  ⚠️  {point_id}: {e}")

In [ ]:
# ── Summary ────────────────────────────────
log = manifest.load_log(LOG_PATH)
n_remaining = len(manifest.pending_items(all_point_ids, TVG_OUT_DIR, ext=".pt"))
print(f"Remaining pending after this run: {n_remaining} / {len(all_point_ids)}")

if "status" in log.columns and (log["status"] == "error").any():
    errors = log[log["status"] == "error"]
    print(f"\n⚠️  {len(errors)} points failed — re-running this notebook will retry them.")
    display(errors[["point_id", "error", "timestamp"]].tail(20))
else:
    print("\n✅ No errors logged.")

In [ ]:
# ── QC: node-count distributions, combined + per-city ─────────
import seaborn as sns
import matplotlib.pyplot as plt

counts = []
sample_ids = [pid for pid in all_point_ids if manifest.is_done(TVG_OUT_DIR, pid, ext=".pt")]
for pid in tqdm(sample_ids, desc="Scanning graph sizes"):
    d = torch.load(TVG_OUT_DIR / f"{pid}.pt", weights_only=False)
    counts.append({
        "point_id": pid,
        "city": row_lookup.loc[pid, "city"],
        "n_buildings": d["building"].x.shape[0],
        "n_intersections": d["intersection"].x.shape[0],
        "n_peers": d["peer_incident"].x.shape[0],
    })

counts_df = pd.DataFrame(counts)
print(counts_df.drop(columns="city").describe())
print()
display(counts_df.groupby("city")[["n_buildings", "n_intersections", "n_peers"]].mean())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["n_buildings", "n_intersections", "n_peers"]):
    sns.histplot(counts_df[col], bins=20, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(INTERIM_DIR / "qc_tvg_node_count_distribution.png", dpi=150)
plt.show()

n_zero_buildings = (counts_df["n_buildings"] == 0).sum()
if n_zero_buildings:
    print(f"\n⚠️  {n_zero_buildings} points have ZERO included buildings — expected given "
          f"the 50m isovist radius in lower-density areas, but worth spot-checking "
          f"{VIZ_OUT_DIR} if this is a large fraction of the dataset.")

n_zero_intersections = (counts_df["n_intersections"] == 0).sum()
if n_zero_intersections:
    print(f"⚠️  {n_zero_intersections} points have ZERO included intersections — expected "
          f"more often now that u/v are no longer force-included (see notebook intro); "
          f"worth spot-checking if this is a large fraction of the dataset.")

In [ ]:
# ── QC: node-count distributions (positive / negative only) ─────
# Uses reconciled's own 'class' column, not point_id.startswith(...) --
# point_id is city-prefixed now (e.g. "bog_positive_123"), so a plain
# prefix match would silently classify everything as neither.
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="pastel")

class_lookup = dict(zip(reconciled["point_id"], reconciled["class"]))

def scan_node_counts(point_ids, desc="Scanning graph sizes"):
    counts = []
    for pid in tqdm(point_ids, desc=desc):
        data = torch.load(TVG_OUT_DIR / f"{pid}.pt", weights_only=False)
        counts.append({
            "point_id": pid,
            "n_buildings": data["building"].x.shape[0],
            "n_intersections": data["intersection"].x.shape[0],
            "n_peers": data["peer_incident"].x.shape[0],
        })
    return pd.DataFrame(counts)


def plot_node_count_distribution(counts_df, label, color, out_path):
    print(f"\n=== {label} (n={len(counts_df)}) ===")
    print(counts_df.describe())

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    cols = [("n_buildings", "Buildings per point"),
            ("n_intersections", "Intersections per point"),
            ("n_peers", "Peer incidents per point")]
    for ax, (col, xlabel) in zip(axes, cols):
        sns.histplot(data=counts_df, x=col, bins=30, kde=True, color=color, ax=ax)
        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel("Count", fontsize=10)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()

    n_zero = (counts_df[["n_buildings", "n_intersections", "n_peers"]].sum(axis=1) == 0).sum()
    if n_zero:
        print(f"\n⚠️  [{label}] {n_zero} points have ZERO nodes across all TVG node types — "
              f"expected occasionally, but worth spot-checking their visualizations "
              f"in {VIZ_OUT_DIR} if this is a large fraction of the dataset.")


positive_ids = [pid for pid in sample_ids if class_lookup.get(pid) == "positive"]
negative_ids = [pid for pid in sample_ids if class_lookup.get(pid) == "negative"]
print(f"Total: {len(sample_ids)} | Positive: {len(positive_ids)} | Negative: {len(negative_ids)}")

counts_df_pos = scan_node_counts(positive_ids, desc="Scanning graph sizes (positive)")
plot_node_count_distribution(
    counts_df_pos, "positive", "red", INTERIM_DIR / "qc_tvg_node_count_distribution_positive.png"
)

counts_df_neg = scan_node_counts(negative_ids, desc="Scanning graph sizes (negative)")
plot_node_count_distribution(
    counts_df_neg, "negative", "blue", INTERIM_DIR / "qc_tvg_node_count_distribution_negative.png"
)

In [ ]:
print(f"TVG graphs saved to: {TVG_OUT_DIR}")
print(f"QC maps saved to: {VIZ_OUT_DIR}")
print()
print("Next: 05_dataset_assembly.ipynb")